# agent-decision-3b: snap decisions about agent tool calls

[`grounded-ai/agent-decision-3b`](https://huggingface.co/grounded-ai/agent-decision-3b) is a fine-tune of Liquid AI's [d1-3B](https://huggingface.co/LiquidAI/d1-3B) decision model, the 3B sibling of [agent-decision-9b](https://huggingface.co/grounded-ai/agent-decision-9b). Given the tools, the user's request and the agent's proposed tool calls, it answers typed questions (*are these calls correct?*, *what is wrong with them?*, *should the agent call a tool, ask, or decline?*) with a calibrated probability for every answer, in one forward pass and without generating text.

Use it anywhere an agent's step needs a fast, calibrated check: as a hook that vets tool calls before they run (like the grounding check in [`grounded_ai`](https://github.com/grounded-ai/grounded_ai)'s Claude Code plugin), as a guardrail that stops an agent from acting on a confidently wrong call, for labelling or filtering agent traces, or as the cheap first stage of an evaluation cascade where only low-confidence decisions go to an LLM judge.

This notebook:

1. shows a few rows of the test data,
2. scores the released d1-3B on a sample of the test set,
3. loads agent-decision-3b and scores the same rows,
4. compares the two, and runs one decision of your own.

**Runtime:** a GPU with bfloat16: **L4**, A100 or H100 (Runtime → Change runtime type). The model is about 6 GB.

In [ ]:
# Colab ships torchao 0.10, which recent PEFT refuses; we do not use it.
%pip uninstall -y -q torchao
%pip install -q "transformers>=5.14" "peft>=0.18" huggingface_hub pandas

In [ ]:
import json, random, sys
import pandas as pd
import torch
from huggingface_hub import hf_hub_download, snapshot_download

assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "needs a GPU with bfloat16 (L4, A100, H100)"
print(torch.cuda.get_device_name())
pd.set_option("display.max_colwidth", 120)

NVIDIA L4


## 1. The data

The test set ([`grounded-ai/agent-decision-eval`](https://huggingface.co/datasets/grounded-ai/agent-decision-eval)) holds 1,139 rows from APIs never seen in training, with a fixed number of rows per kind of example. Every label is exact:

- **Tool calls** come from [xLAM function-calling](https://huggingface.co/datasets/Salesforce/xlam-function-calling-60k), whose calls were verified by execution. Half are shown as they are. The rest carry exactly one edit that makes them wrong: a tool swapped for one whose schema rejects the arguments, a parameter the tool does not define, a value the user typed removed or changed to one they did not type, or one of several calls dropped.
- **Call, ask or decline** comes from [NVIDIA When2Call](https://huggingface.co/datasets/nvidia/When2Call)'s curated answers.

In [ ]:
path = hf_hub_download("grounded-ai/agent-decision-eval", "test.jsonl", repo_type="dataset")
TEST = [json.loads(line) for line in open(path)]
kind = lambda r: f'{r["source"]}:{r["edit"] or r["kind"]}'
print(len(TEST), "test rows")
print(pd.Series([kind(r) for r in TEST]).value_counts().to_string())

1139 test rows
xlam:swapped_tool             150
xlam:correct                  150
xlam:undefined_parameter      150
xlam:changed_value            144
when2call:cannot_answer       126
when2call:tool_call           126
when2call:request_for_info    111
xlam:missing_value            100
xlam:dropped_call              82


In [ ]:
def show(row, max_chars=1400):
    state = row["state"] if isinstance(row["state"], str) else json.dumps(row["state"], ensure_ascii=False, indent=1)
    print("=" * 100)
    print(kind(row))
    print(state if len(state) <= max_chars else state[:max_chars] + "\n...")
    for qid, q in row["questions"].items():
        print(f'\nQ: {q["instructions"]}')
        if q.get("criteria"):
            print("   options:", list(q["criteria"]))
        print("   gold:", row["gold"][qid])


# One example of every kind, fixed up front (the first transcript-style row of each kind), not picked by result.
EXAMPLES = {k: next(r for r in TEST if kind(r) == k and r["style"] == "transcript") for k in sorted({kind(r) for r in TEST})}
for k in ("xlam:changed_value", "xlam:undefined_parameter", "xlam:correct", "when2call:request_for_info"):
    show(EXAMPLES[k])

xlam:changed_value
TOOLS:
{"name": "v1", "description": "Retrieves filtered observations from the Version 1 endpoint based on specified time and filters.", "parameters": {"time": {"description": "Select one of the following timeframes:", "type": "str", "default": "today"}, "filter1": {"description": "Can be a specific user, type, or tag (e.g., url, domain, ip, sha256, md5 for type, and phishing, ransomware, CobaltStrike for tag). Default is 'phishing'.", "type": "str, optional", "default": "phishing"}, "filter2": {"description": "Can be a specific user, type, or tag (e.g., url, domain, ip, sha256, md5 for type, and phishing, ransomware, CobaltStrike for tag). Default is 'url'.", "type": "str, optional", "default": "url"}}}
{"name": "query", "description": "Queries information about an IP address using the NetDetective API.", "parameters": {"ipaddress": {"description": "The IP address to use for the query. Defaults to None, which uses the IP address of the connecting client.", "type": "

## 2. The released d1-3B, zero-shot

We score a sample of 20 rows per kind of example (180 rows) so the notebook runs in minutes, plus the nine example rows. d1 is a strong starting point: on the full test set the released model gets **79.5%** of decisions right.

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained("LiquidAI/d1-3B", revision="051bcc464b01b9f92942b364d9586b0ef5912432",
                                  trust_remote_code=True, dtype=torch.bfloat16).to("cuda")

rng = random.Random(0)
SAMPLE = []
for k in sorted({kind(r) for r in TEST}):
    group = [r for r in TEST if kind(r) == k]
    SAMPLE += rng.sample(group, min(20, len(group)))
print(len(SAMPLE), "rows in the sample")

180 rows in the sample


In [ ]:
def gold_key(question, gold):
    return ("true" if gold else "false") if question["type"] == "noul" else str(gold)


FAMILY_NAMES = {"tool_call_ok": "calls correct?", "tool_call_kind": "what is wrong?", "call_ask_decline": "call / ask / decline"}


def picked(question, answer):
    """The option a SystemOne answer picks, and the probability it gives that option."""
    if question["type"] == "noul":
        return ("true" if answer["noul"] >= 0.5 else "false"), max(answer["noul"], 1 - answer["noul"])
    if question["type"] == "choice":
        return answer["choice"], answer["confidence"]
    best = max(answer["probabilities"], key=answer["probabilities"].get)
    return best, answer["probabilities"][best]


def score(rows, batch_size=32):
    """One row per question: the model's answer, its confidence, and whether it matches the gold answer."""
    out = []
    for i in range(0, len(rows), batch_size):
        chunk = rows[i : i + batch_size]
        for row, response in zip(chunk, model.system_one_batch([(r["state"], r["questions"]) for r in chunk])):
            for qid, question in row["questions"].items():
                answer, confidence = picked(question, response["answers"][qid])
                gold = gold_key(question, row["gold"][qid])
                out.append({"id": row["id"], "question": FAMILY_NAMES[row["families"][qid]], "kind": kind(row),
                            "gold": gold, "answer": answer, "confidence": confidence, "correct": answer == gold})
    return pd.DataFrame(out)


base = score(SAMPLE)
base_examples = score(list(EXAMPLES.values()))
print(f'd1-3B: {base["correct"].mean():.1%} correct; '
      f'{(base["confidence"] >= 0.9).mean():.0%} answered with 90%+ confidence, of which {base[base["confidence"] >= 0.9]["correct"].mean():.1%} correct')
base.groupby("question")["correct"].mean().map("{:.1%}".format)

d1-3B: 80.8% correct; 39% answered with 90%+ confidence, of which 93.5% correct


,correct
question,
call / ask / decline,73.3%
calls correct?,85.0%
what is wrong?,79.7%


## 3. agent-decision-3b

The fine-tune is a LoRA adapter on d1's language model plus `calibration.json`, a temperature applied to the option scores at readout. These are the loading lines from the model card.

In [ ]:
from peft import PeftModel

ours = snapshot_download("grounded-ai/agent-decision-3b")
model = PeftModel.from_pretrained(model, ours).merge_and_unload()


class Temperature:
    """Divides each option's score by the fitted temperature before the softmax."""
    def __init__(self, t):
        self.t = t

    def apply(self, question, scores):
        return [s / self.t for s in scores]


model.engine.calibration = Temperature(json.load(open(f"{ours}/calibration.json"))["temperature"])

tuned = score(SAMPLE)
tuned_examples = score(list(EXAMPLES.values()))
print(f'agent-decision-3b: {tuned["correct"].mean():.1%} correct; '
      f'{(tuned["confidence"] >= 0.9).mean():.0%} answered with 90%+ confidence, of which {tuned[tuned["confidence"] >= 0.9]["correct"].mean():.1%} correct')

agent-decision-3b: 95.4% correct; 84% answered with 90%+ confidence, of which 99.0% correct


## 4. Before and after, on the same rows

In [ ]:
def table(by):
    t = pd.DataFrame({"d1-3B": base.groupby(by)["correct"].mean(), "agent-decision-3b": tuned.groupby(by)["correct"].mean()})
    t.loc["all"] = [base["correct"].mean(), tuned["correct"].mean()]
    return t.map("{:.1%}".format)


table("question")

,d1-3B,agent-decision-3b
question,,
call / ask / decline,73.3%,88.3%
calls correct?,85.0%,98.3%
what is wrong?,79.7%,96.6%
all,80.8%,95.4%


In [ ]:
# "Are the calls correct?" by kind of call: which mistakes each model catches.
ok_base, ok_tuned = base[base["question"] == "calls correct?"], tuned[tuned["question"] == "calls correct?"]
pd.DataFrame({"d1-3B": ok_base.groupby("kind")["correct"].mean(),
              "agent-decision-3b": ok_tuned.groupby("kind")["correct"].mean()}).map("{:.1%}".format)

,d1-3B,agent-decision-3b
kind,,
xlam:changed_value,85.0%,95.0%
xlam:correct,100.0%,100.0%
xlam:dropped_call,65.0%,95.0%
xlam:missing_value,95.0%,100.0%
xlam:swapped_tool,100.0%,100.0%
xlam:undefined_parameter,65.0%,100.0%


In [ ]:
# One example of every kind, answered by both models (the four shown in section 1 are among them).
side = base_examples.merge(tuned_examples, on=["id", "question", "kind", "gold"], suffixes=(" (base)", " (ours)"))
side[["kind", "question", "gold", "answer (base)", "confidence (base)", "answer (ours)", "confidence (ours)"]].round(3)

,kind,question,gold,answer (base),confidence (base),answer (ours),confidence (ours)
0,when2call:cannot_answer,call / ask / decline,cannot_answer,cannot_answer,0.865,cannot_answer,0.981
1,when2call:request_for_info,call / ask / decline,ask,call,0.890,call,0.487
2,when2call:tool_call,call / ask / decline,call,call,0.807,call,0.674
3,xlam:changed_value,calls correct?,false,true,0.500,true,0.606
4,xlam:changed_value,what is wrong?,wrong_arguments,wrong_arguments,0.350,correct,0.662
5,xlam:correct,calls correct?,true,true,0.980,true,0.998
6,xlam:correct,what is wrong?,correct,correct,0.967,correct,0.995
7,xlam:dropped_call,calls correct?,false,true,0.562,false,0.980
8,xlam:missing_value,calls correct?,false,false,0.940,false,1.000
9,xlam:missing_value,what is wrong?,wrong_arguments,wrong_arguments,0.749,wrong_arguments,0.996


## 5. Ask your own question

The model answers through d1's `system_one`: a `state` (any JSON or text) and typed `questions`. Here the user asked for the account `guest`, and the agent called the login tool with `guest789`.

In [ ]:
tools = [{"name": "loginuser", "description": "Log in a user.",
          "parameters": {"username": {"type": "str"}, "password": {"type": "str"}}}]
state = {"tools": tools, "user": "Log in a user named 'guest' with the password 'guestpass'.",
         "assistant_tool_calls": [{"name": "loginuser", "arguments": {"username": "guest789", "password": "guestpass"}}]}
response = model.system_one(state, {
        "call_ok": {"type": "noul", "instructions": "Are the assistant's tool calls correct and complete for the user's request?"},
        "call_kind": {"type": "choice", "instructions": "What, if anything, is wrong with the proposed tool calls?",
                      "criteria": {"correct": "the calls are right and do everything the user asked",
                                   "wrong_tool": "a call uses a tool that cannot do what was asked",
                                   "wrong_arguments": "the right tool is called with arguments that do not match the request or the tool",
                                   "incomplete": "the calls are right but part of the request is not done"}},
})
print(json.dumps(response["answers"], indent=2))

{
  "call_ok": {
    "type": "noul",
    "noul": 0.00013517050793600177
  },
  "call_kind": {
    "type": "choice",
    "choice": "wrong_arguments",
    "confidence": 0.9945801831496567,
    "probabilities": {
      "correct": 0.0029185328983726746,
      "wrong_tool": 0.0008897951077937156,
      "wrong_arguments": 0.9945801831496567,
      "incomplete": 0.0016114888441770296
    }
  }
}


## Full test results

On all 1,139 test rows (from the training run, same rows for every model):

| Question | d1-3B | agent-decision-3b | agent-decision-9b |
|---|---|---|---|
| Are the calls correct? | 86.2% | **98.3%** | 98.1% |
| What is wrong with them? | 75.3% | **96.4%** | 96.4% |
| Call, ask or decline? | 69.7% | **85.4%** | 88.2% |
| All | 79.5% | **94.8%** | 95.3% |

86% of agent-decision-3b's answers come with 90%+ confidence, and 99.1% of those are right. Calibration error 0.011.

**Limitations.** Wrong calls in training and test are exact edits of correct calls: certain to be wrong, but possibly easier to spot than real agent mistakes. "Ask the user for missing information" is the weakest decision (68.5% here, 52% in an unseen markdown layout; the 9B gets 81%). A call with a parameter the tool does not define counts as wrong.

Model: [grounded-ai/agent-decision-3b](https://huggingface.co/grounded-ai/agent-decision-3b) (LFM Open License v1.0) · Data: [grounded-ai/agent-decision-eval](https://huggingface.co/datasets/grounded-ai/agent-decision-eval) · Base: [LiquidAI/d1-3B](https://huggingface.co/LiquidAI/d1-3B) · Training data: xLAM function-calling (Salesforce), When2Call (NVIDIA), both CC-BY-4.0.